In [ ]:
import pandas as pd
from pathlib import Path
import zipfile

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR.parents[1] / "project_data"

# Step 1 读取数据

In [ ]:
entries_en_path  = DATA_DIR / 'entries1_en.zip'

entries_en = pd.read_json(
    entries_en_path,   
    orient="records",
    lines=True,
    compression="zip"
)

In [ ]:
def read_tsv_from_zip(zip_path, inner_name, columns=None):
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(inner_name) as f:
            return pd.read_csv(
                f,
                sep="\t",              # FriendFeed 是 TSV
                names=columns,
                header=None if columns else "infer",
                na_values="\\N",
                engine="python",
                encoding="utf-8",
                on_bad_lines="skip"
            )

In [ ]:
comments_cols = [
    "PostID",
    "EntryID",
    "PostedBy",
    "SourceName",
    "SourceURL",
    "GeoX",
    "GeoY",
    "Timestamp",
    "Text",
    "NumImg",
    "ImgURL",
    "NumVid",
    "VideoURL",
]

comments = read_tsv_from_zip(
    DATA_DIR / "comments.zip",
    "commentAugSept.csv",          # zip 里真实文件名
    columns=comments_cols
)

In [ ]:
likes_cols = [
    "UserID",
    "PostID",
    "Timestamp",
]
likes = read_tsv_from_zip(
    DATA_DIR / "likes.zip", 
    "likes.csv",          # zip 里真实文件名
    columns=likes_cols
)

In [ ]:
import csv

users_cols = [
    "UserID",
    "Type",
    "Name",
    "Reserved",
    "Description"]

users_zip = DATA_DIR / "users.zip"

with zipfile.ZipFile(users_zip, "r") as z:
    print(z.namelist())  # 先确认文件名

    with z.open("users.csv") as f:
        users = pd.read_csv(
            f,
            sep="|",                  # ⭐ 关键：竖线分隔
            names=users_cols,
            header=None,
            na_values=["\\N", "null"],
            engine="python",
            encoding="utf-8",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE
        )


users.head()


# Step 2 创建无相图

In [ ]:
import networkx as nx
G = nx.Graph() 
post_author_map = (
    entries_en[["PostID", "PostedBy"]]
    .drop_duplicates()
    .set_index("PostID")["PostedBy"]
    .to_dict())

# Step 3 统计评论次数（无相）

In [ ]:
comment_pairs = []

for _, row in comments.iterrows():
    commenter = row["PostedBy"]
    post_id = row["EntryID"]

    # 只保留英文帖子
    if post_id not in post_author_map:
        continue

    post_author = post_author_map[post_id]

    # 去掉自评论
    if commenter == post_author:
        continue

    # 无向：排序保证 (u, v) 和 (v, u) 视为同一条边
    u, v = sorted([commenter, post_author])
    comment_pairs.append((u, v))

comment_df = (
    pd.DataFrame(comment_pairs, columns=["u", "v"])
    .value_counts()
    .reset_index(name="comment_count")
)
print(f"Unique comment edges: {comment_df.shape[0]}")

# Step 4 统计点赞次数（无相）

In [ ]:
like_pairs = []

for _, row in likes.iterrows():
    liker = row["UserID"]
    post_id = row["PostID"]

    if post_id not in post_author_map:
        continue

    post_author = post_author_map[post_id]

    # 去掉自赞
    if liker == post_author:
        continue

    u, v = sorted([liker, post_author])
    like_pairs.append((u, v))
like_df = (
    pd.DataFrame(like_pairs, columns=["u", "v"])
    .value_counts()
    .reset_index(name="like_count")
)


# Step 5 加入边属性

In [ ]:
edge_df = pd.merge(
    comment_df,
    like_df,
    on=["u", "v"],
    how="outer"
).fillna(0)

edge_df["comment_count"] = edge_df["comment_count"].astype(int)
edge_df["like_count"] = edge_df["like_count"].astype(int)
print(f"Total unique edges (comments + likes): {edge_df.shape[0]}")

In [ ]:
for _, r in edge_df.iterrows():
    G.add_edge(
        r["u"], r["v"],
        comment_count=int(r["comment_count"]),
        like_count=int(r["like_count"]),
        weight=int(r["comment_count"]) + int(r["like_count"])
    )

In [ ]:
users = users.drop_duplicates(subset=["UserID"]).copy()
node_ids = list(G.nodes())

user_attrs_df = (
    users[users["UserID"].isin(node_ids)]
    .set_index("UserID")[["Type", "Name", "Description"]]   
)

node_attr_dict = user_attrs_df.to_dict("index")
nx.set_node_attributes(G, node_attr_dict)


In [ ]:
list(G.edges(data=True))[:5]


In [ ]:
import pickle

file_path_pkl = DATA_DIR / 'network_data.pkl'
with open(file_path_pkl, 'wb') as f: # 注意 'wb' (写入二进制)
    pickle.dump(G, f)